# CLI & Agent Skills Changelog

> Readable companion to `debug/cli_and_skills_changelog.md` — same content, notebook form.

**Branch:** `feature/hydra-e2e-cli`  
**PR:** [#5](https://github.com/HasanGoni/vision_ad_tool/pull/5)  
**Base:** `main`

## Executive summary

This branch adds three layers on top of `be_vision_ad_tools`:

1. **7 local Hydra CLIs** (`vad-train` … `vad-hyperparam-search`) — nbdev-exported, YAML-configured
2. **7 HPC submit CLIs** (`vad-*-submit`) — wrap `bsub`, run the local CLI on the office LSF cluster
3. **15 agent skills** — 8 workflow + 7 HPC submit, pyskills-aligned (`.cursor/skills/` + `pyskills-bridge/`)

See the markdown file for full tables and usage examples.

## 1. Hydra local CLIs (nbdev way)

### Why

Make every AD pipeline **reproducible** and **agent-friendly**: one YAML default per workflow, Hydra overrides on the CLI, notebooks as source of truth.

### Notebooks → modules

| Notebook | Exported module | CLIs |
|----------|-----------------|------|
| `nbs/18_tutorials.end2end_hydra_cli.ipynb` | `be_vision_ad_tools/tutorials/end2end_cli.py` | `vad-train`, `vad-infer`, `vad-organize`, `vad-infer-organize`, `vad-train-infer`, `vad-full` |
| `nbs/19_training.hyperparameter_hydra_cli.ipynb` | `be_vision_ad_tools/training/hyperparameter_hydra_cli.py` | `vad-hyperparam-search` |

### Configs

`be_vision_ad_tools/tutorials/conf/*.yaml` — packaged via `pyproject.toml` package-data.

### Usage

```bash
uv sync
vad-train data_root=/path/to/data model_name=patchcore
vad-full train.data_root=/data/train infer_organize.test_images=/data/test
```

### nbdev v3

- Export: `uv run nbdev-export` (hyphenated, not `nbdev_export`)
- Test: `uv run nbdev-test`

### Legacy shims

`tutorials/end2end_tutorial/` — thin re-exports for backward compatibility.

## Master mapping: CLI → module → notebook → skill → config

| CLI | Module | Notebook | Cursor skill | Config |
|-----|--------|----------|--------------|--------|
| `vad-train` | `end2end_cli.train_cli` | 18 | `train-anomaly-model` | `train.yaml` |
| `vad-infer` | `end2end_cli.infer_cli` | 18 | `run-ad-inference` | `infer.yaml` |
| `vad-organize` | `end2end_cli.organize_cli` | 18 | `organize-anomaly-scores` | `organize.yaml` |
| `vad-infer-organize` | `end2end_cli.infer_organize_cli` | 18 | `infer-organize-ad` | `infer_organize.yaml` |
| `vad-train-infer` | `end2end_cli.train_infer_cli` | 18 | `train-infer-ad` | `train_infer.yaml` |
| `vad-full` | `end2end_cli.full_cli` | 18 | `full-ad-pipeline` | `full.yaml` |
| `vad-hyperparam-search` | `hyperparameter_hydra_cli.hyperparam_search_cli` | 19 | `hyperparameter-search-ad` | `hyperparam_search.yaml` |

## 2. HPC submit CLIs

### Source

`nbs/20_tutorials.hpc_submit_cli.ipynb` → `be_vision_ad_tools/tutorials/hpc_submit_cli.py`

### Submit mapping

| Submit CLI | Local CLI | Config overlay | Cursor skill |
|------------|-----------|----------------|--------------|
| `vad-train-submit` | `vad-train` | `train_submit.yaml` | `submit-train-ad-hpc` |
| `vad-infer-submit` | `vad-infer` | `infer_submit.yaml` | `submit-infer-ad-hpc` |
| `vad-organize-submit` | `vad-organize` | `organize_submit.yaml` | `submit-organize-ad-hpc` |
| `vad-infer-organize-submit` | `vad-infer-organize` | `infer_organize_submit.yaml` | `submit-infer-organize-ad-hpc` |
| `vad-train-infer-submit` | `vad-train-infer` | `train_infer_submit.yaml` | `submit-train-infer-ad-hpc` |
| `vad-full-submit` | `vad-full` | `full_submit.yaml` | `submit-full-ad-hpc` |
| `vad-hyperparam-search-submit` | `vad-hyperparam-search` | `hyperparam_search_submit.yaml` | `submit-hyperparam-search-ad-hpc` |

Shared HPC defaults: `conf/hpc/submit_defaults.yaml`

### Usage

```bash
# Preview without bsub
vad-train-submit data_root=/data/my_product hpc.dry_run=true

# GPU submit
vad-train-submit data_root=/data/my_product hpc.use_gpu=true hpc.queue=gpu
```

## 3. Agent skills (pyskills steal pattern)

Pattern from `docs/agent-skills-from-pyskills.md`:

| Step | Meaning |
|------|--------|
| **DISCOVER** | YAML `description:` — when to load the skill |
| **CURATE** | Public API table + `scripts/` only |
| **DEMONSTRATE** | "What done looks like" with concrete numbers |
| **EXECUTE** | Shell scripts, not inline bash |
| **BOUND** | "Do not invent other commands" |

### Locations

- **Cursor:** `.cursor/skills/<name>/SKILL.md` + `scripts/`
- **pyskills:** `pyskills-bridge/<module>/skill.py` with `__all__`, `allow()`
- **Reference:** `pyskills-bridge/REFERENCE.md`

15 skills total: 8 workflow + 7 HPC submit (+ `verify-ad-pipeline`).

## 4. Other notable fixes

- **`create_posters_for_score_folders`** — re-exported from notebook 14 (`6d8d0a9`); fixes `vad-infer-organize` posters
- **nbdev3 migration** — `[tool.nbdev]` in `pyproject.toml`, hyphenated CLI names, configs under library path
- **`AGENTS.md`** — HPC conventions, new-workflow checklist, verification, preprocessing contract

## 5. Commits on this branch

| Commit | Summary |
|--------|--------|
| `9ac1445` | Initial Hydra CLIs + nbdev3 pyproject |
| `fa855b0` | Full pyskills-aligned agent skills |
| `93aa2ac` | nbdev-native export + `vad-hyperparam-search` |
| `828f64e` | Fix nbdev v3 CLI names + import paths |
| `6d8d0a9` | Export `create_posters_for_score_folders` |
| `839358d` | Align skill docs with pyskills audit |
| `7cfe5da` | HPC submit CLIs + skills + pyskills twins |
| `f11fa83` | Smoke-test `vad-train-submit` in verify pipeline |

## Verify before commit

```bash
bash .cursor/skills/verify-ad-pipeline/scripts/verify_sync.sh
bash .cursor/skills/verify-ad-pipeline/scripts/verify_imports.sh
bash .cursor/skills/verify-ad-pipeline/scripts/verify_nbdev.sh
```

Full details: `debug/cli_and_skills_changelog.md`